In [1]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Experimentation with Transformer Time-Series Models



In [ ]:
import importlib
import time
from pathlib import Path

import numpy as np
import pandas as pd

from RADAR.time_series.algorithms import transformers
from RADAR.time_series.preprocessing.preprocessing_ts import StandardScalerPreprocessing
from RADAR.time_series.time_series_datasets_uci import global_load as load_time_series
from RADAR.time_series.time_series_utils import TimeSeriesProcessor
import RADAR.metrics_module as metrics_module

metrics_module = importlib.reload(metrics_module)

## Build the UCI Time-Series Benchmark

- `ai4i_2020_predictive_maintenance_dataset`: anomaly labels come directly from `Machine failure`.
- `metro_interstate_traffic_volume`: anomaly labels are derived from extreme traffic volume values using the 5th and 95th percentiles.


In [ ]:
WINDOW_SIZE = 24
STEP_SIZE = 1
TEST_SIZE = 0.2
METRO_LOW_Q = 0.05
METRO_HIGH_Q = 0.95

def chronological_split(X, y, test_size=0.2):
    split_idx = int(len(X) * (1 - test_size))
    return X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

def aggregate_window_labels(y_windows):
    y_windows = np.asarray(y_windows)
    if y_windows.ndim == 1:
        return y_windows.astype(int)
    return (y_windows.sum(axis=1) > 0).astype(int)

def prepare_ai4i_dataset(window_size=WINDOW_SIZE, step_size=STEP_SIZE, test_size=TEST_SIZE):
    X, y = load_time_series('ai4i_2020_predictive_maintenance_dataset')
    labels = y['Machine failure'].astype(int).to_numpy()
    X = X.drop(columns=['Type'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)

    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(X_train, y_train, X_test, y_test)

    return {
        'dataset': 'ai4i_2020_predictive_maintenance_dataset',
        'X_train_windows': np.asarray(X_train_windows, dtype=np.float32),
        'X_test_windows': np.asarray(X_test_windows, dtype=np.float32),
        'y_test_windows': np.asarray(y_test_windows),
        'y_test_labels': aggregate_window_labels(y_test_windows),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'window_size': window_size,
        'train_windows': len(X_train_windows),
        'test_windows': len(X_test_windows),
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'positive_ratio_windows': round(float(np.mean(aggregate_window_labels(y_test_windows))), 4),
        'label_note': 'Machine failure from UCI target',
    }

def prepare_metro_dataset(window_size=WINDOW_SIZE, step_size=STEP_SIZE, test_size=TEST_SIZE, low_q=METRO_LOW_Q, high_q=METRO_HIGH_Q):
    X, y = load_time_series('metro_interstate_traffic_volume')
    traffic_volume = y['traffic_volume'].astype(float)
    low_threshold = float(traffic_volume.quantile(low_q))
    high_threshold = float(traffic_volume.quantile(high_q))
    labels = ((traffic_volume <= low_threshold) | (traffic_volume >= high_threshold)).astype(int).to_numpy()

    X = X.drop(columns=['date_time', 'holiday', 'weather_main', 'weather_description'], errors='ignore')

    scaler = StandardScalerPreprocessing()
    X_scaled = scaler.fit_transform(X)
    X_values = np.asarray(X_scaled, dtype=np.float32)

    X_train, X_test, y_train, y_test = chronological_split(X_values, labels, test_size=test_size)

    processor = TimeSeriesProcessor(window_size=window_size, step_size=step_size, future_prediction=False)
    X_train_windows, y_train_windows, X_test_windows, y_test_windows = processor.process_train_test(X_train, y_train, X_test, y_test)

    return {
        'dataset': 'metro_interstate_traffic_volume',
        'X_train_windows': np.asarray(X_train_windows, dtype=np.float32),
        'X_test_windows': np.asarray(X_test_windows, dtype=np.float32),
        'y_test_windows': np.asarray(y_test_windows),
        'y_test_labels': aggregate_window_labels(y_test_windows),
        'n_samples': len(X_values),
        'n_features': X_values.shape[1],
        'window_size': window_size,
        'train_windows': len(X_train_windows),
        'test_windows': len(X_test_windows),
        'positive_ratio_points': round(float(np.mean(labels)), 4),
        'positive_ratio_windows': round(float(np.mean(aggregate_window_labels(y_test_windows))), 4),
        'label_note': f'Extreme traffic volume: <= q{low_q:.2f} or >= q{high_q:.2f}',
        'low_threshold': round(low_threshold, 3),
        'high_threshold': round(high_threshold, 3),
    }

dataset_configs = {
    'ai4i': prepare_ai4i_dataset(),
    'metro_interstate': prepare_metro_dataset(),
}

In [ ]:
dataset_summary = pd.DataFrame([
    {
        'dataset_key': dataset_key,
        'dataset_name': config['dataset'],
        'samples': config['n_samples'],
        'features': config['n_features'],
        'window_size': config['window_size'],
        'train_windows': config['train_windows'],
        'test_windows': config['test_windows'],
        'positive_ratio_points': config['positive_ratio_points'],
        'positive_ratio_windows': config['positive_ratio_windows'],
        'label_note': config['label_note'],
    }
    for dataset_key, config in dataset_configs.items()
]).reset_index(drop=True)

display(dataset_summary)

In [ ]:
def tensor_to_numpy(values):
    if hasattr(values, 'detach'):
        values = values.detach().cpu().numpy()
    return np.asarray(values)

def summarize_mse(scores):
    scores = np.asarray(scores, dtype=float).ravel()
    return float(scores.mean()) if scores.size else np.nan

transformer_model_configs = [
    {
        'algorithm_': 'transformer',
        'd_model': 64,
        'd_qk': 64,
        'd_v': 64,
        'n_layers': 2,
        'n_heads': 8,
        'ulayers_feedfwd': 128,
        'dropout_rate': 0.1,
        'attns_outs': False,
        'train_epochs': 5,
        'batch_size': 32,
        'lr': 1e-3,
    },
    {
        'algorithm_': 'informer',
        'd_model': 64,
        'n_heads': 8,
        'e_layers': 2,
        'd_layers': 1,
        'd_ff': 128,
        'factor': 5,
        'dropout': 0.1,
        'attn': 'prob',
        'activation': 'gelu',
        'output_attention': False,
        'distil': True,
        'mix': True,
        'train_epochs': 5,
        'batch_size': 32,
        'lr': 1e-3,
    },
    {
        'algorithm_': 'autoformer',
        'd_model': 64,
        'n_heads': 8,
        'e_layers': 2,
        'd_layers': 1,
        'd_ff': 128,
        'factor': 5,
        'moving_avg': 5,
        'dropout': 0.1,
        'activation': 'gelu',
        'output_attention': False,
        'train_epochs': 5,
        'batch_size': 32,
        'lr': 1e-3,
    },
]

transformer_results = []

for dataset_key, config in dataset_configs.items():
    input_dim = config['X_train_windows'].shape[2]
    seq_len = config['window_size']

    print(f'\nDataset: {config["dataset"]}')
    print(f'Features: {input_dim} | Train windows: {config["train_windows"]} | Test windows: {config["test_windows"]}')

    for model_template in transformer_model_configs:
        model_params = dict(model_template)
        algorithm_name = model_params['algorithm_']

        if algorithm_name == 'transformer':
            model_params.update({
                'label_parser': None,
                'size_enc_in': input_dim,
                'size_dec_in': input_dim,
                'seq_len': seq_len,
            })
        elif algorithm_name == 'informer':
            model_params.update({
                'label_parser': None,
                'enc_in': input_dim,
                'dec_in': input_dim,
                'c_out': input_dim,
                'seq_len': seq_len,
                'label_len': seq_len,
                'out_len': seq_len,
            })
        elif algorithm_name == 'autoformer':
            model_params.update({
                'label_parser': None,
                'enc_in': input_dim,
                'dec_in': input_dim,
                'c_out': input_dim,
                'seq_len': seq_len,
                'label_len': seq_len,
                'pred_len': seq_len,
            })

        model = transformers.TransformersAnomalyDetection(**model_params)

        train_start = time.time()
        model.fit(config['X_train_windows'])
        train_time = time.time() - train_start

        inference_start = time.time()
        scores = tensor_to_numpy(model.decision_function(config['X_test_windows'])).ravel()
        inference_time = time.time() - inference_start

        finite_scores = bool(np.isfinite(scores).all())
        mse = summarize_mse(scores) if finite_scores else np.nan

        print(f'  Model: {algorithm_name}')
        if np.isfinite(mse):
            print(f'    MSE={mse:.6f}')
        else:
            print('    MSE=nan (non-finite scores)')

        transformer_results.append({
            'dataset_key': dataset_key,
            'dataset_name': config['dataset'],
            'algorithm': algorithm_name,
            'window_size': seq_len,
            'n_features': input_dim,
            'train_windows': config['train_windows'],
            'test_windows': config['test_windows'],
            'train_time_s': round(train_time, 4),
            'inference_time_s': round(inference_time, 4),
            'mse': round(float(mse), 6) if np.isfinite(mse) else np.nan,
        })

transformer_results_df = pd.DataFrame(transformer_results).sort_values(
    ['dataset_name', 'mse'],
    ascending=[True, True],
    na_position='last',
).reset_index(drop=True)

display(transformer_results_df)

## Per-Dataset Summary



In [ ]:
transformer_summary_df = (
    transformer_results_df.groupby(['dataset_name', 'algorithm'], as_index=False)
    .agg({
        'mse': 'min',
        'train_time_s': 'mean',
        'inference_time_s': 'mean',
    })
    .sort_values(['dataset_name', 'mse'], ascending=[True, True], na_position='last')
    .reset_index(drop=True)
)

display(transformer_summary_df)

In [ ]:
results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

results_main_path = results_dir / 'uci_transformers_results.csv'
results_summary_path = results_dir / 'uci_transformers_summary.csv'

transformer_results_df.to_csv(results_main_path, index=False)
transformer_summary_df.to_csv(results_summary_path, index=False)

print(f'Saved detailed results to: {results_main_path}')
print(f'Saved summary results to: {results_summary_path}')